In [1]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 🚀 **SYNTHETIC DATA GENERATOR v5.1 - CLEAN + RAW (VECTORIZED)**
# MAGIC 
# MAGIC **Based on:** INSERT_3_VALID_RECORDS.py  
# MAGIC **Features:**
# MAGIC - ✅ Configurable clean/raw record counts
# MAGIC - ✅ EXACT same table structure (all 21 columns)
# MAGIC - ✅ 45% female, 30% male, 5% unknown distribution
# MAGIC - ✅ Realistic age groups (babies to seniors)
# MAGIC - ✅ Future timestamps (for incremental load)
# MAGIC - ✅ 100% vectorized (no UDFs)

# COMMAND ----------

import pyspark.sql.functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import time

print("=" * 80)
print("🚀 SYNTHETIC DATA GENERATOR v5.1 - CLEAN + RAW")
print("=" * 80)
print("Target: EXACT match with INSERT_3_VALID_RECORDS.py")
print("=" * 80)

# COMMAND ----------

# MAGIC %md
# MAGIC ## ⚙️ CONFIGURATION

# COMMAND ----------

class Config:
    """Configuration - matching INSERT_3_VALID_RECORDS.py exactly"""
    
    # ===== TARGET TABLE (EXACT match with INSERT_3_VALID_RECORDS) =====
    TARGET_TABLE = "dbo.person"
    
    # ===== ROW COUNTS (CHANGE THESE!) =====
    CLEAN_COUNT = 500   # Number of clean/valid records
    RAW_COUNT = 30        # Number of raw/problematic records
    
    # ===== GENDER DISTRIBUTION (for clean records) =====
    FEMALE_PERCENT = 45    # 45% female (8532)
    MALE_PERCENT = 50      # 50% male (8507)
    UNKNOWN_PERCENT = 5    # 5% unknown (8551)
    
    # ===== AGE DISTRIBUTION =====
    BABY_PERCENT = 8        # 0-2 years
    CHILD_PERCENT = 15      # 3-12 years
    TEEN_PERCENT = 12       # 13-19 years
    YOUNG_ADULT_PERCENT = 25 # 20-35 years
    ADULT_PERCENT = 30      # 36-64 years
    SENIOR_PERCENT = 10     # 65+ years
    
    # ===== RAW RECORD TYPES (matching INSERT_5_RAW_RECORDS.py) =====
    NULL_ID_PERCENT = 20            # NULL person_id
    BAD_GENDER_PERCENT = 25         # Invalid gender code (7777,8888,9999)
    FUTURE_BIRTH_PERCENT = 15       # Future birth year (2028+)
    OLD_YEAR_PERCENT = 20           # Year < 1900 (1700-1899)
    BAD_MONTH_PERCENT = 10          # Month 13-15
    BAD_DAY_PERCENT = 10            # Day 32-35
    
    # ===== TIMESTAMP (EXACT match with INSERT_3_VALID_RECORDS) =====
    # Using future timestamp to ensure incremental load picks them up
    FUTURE_MINUTES = 15
    
    # ===== STARTING PERSON ID =====
    # Clean records: START_ID onward
    # Raw records: START_ID + CLEAN_COUNT onward
    START_ID = 2000000  # Matching pattern from working scripts

# Validate configuration
total_pct = Config.FEMALE_PERCENT + Config.MALE_PERCENT + Config.UNKNOWN_PERCENT
if total_pct != 100:
    print(f"⚠️  Warning: Gender percentages sum to {total_pct}%, should be 100%")

age_total = (Config.BABY_PERCENT + Config.CHILD_PERCENT + Config.TEEN_PERCENT + 
            Config.YOUNG_ADULT_PERCENT + Config.ADULT_PERCENT + Config.SENIOR_PERCENT)
if age_total != 100:
    print(f"⚠️  Warning: Age percentages sum to {age_total}%, should be 100%")

raw_total = (Config.NULL_ID_PERCENT + Config.BAD_GENDER_PERCENT + 
            Config.FUTURE_BIRTH_PERCENT + Config.OLD_YEAR_PERCENT +
            Config.BAD_MONTH_PERCENT + Config.BAD_DAY_PERCENT)
if raw_total != 100:
    print(f"⚠️  Warning: Raw type percentages sum to {raw_total}%, should be 100%")

total_records = Config.CLEAN_COUNT + Config.RAW_COUNT

print("\n📋 CONFIGURATION SUMMARY:")
print(f"   {'='*60}")
print(f"   Target Table:      {Config.TARGET_TABLE}")
print(f"   {'='*60}")
print(f"   CLEAN Records:     {Config.CLEAN_COUNT:,}")
print(f"   RAW Records:       {Config.RAW_COUNT:,}")
print(f"   TOTAL Records:     {total_records:,}")
print(f"   START_ID:          {Config.START_ID:,}")
print(f"   {'='*60}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 📊 TABLE SCHEMA (EXACTLY 21 COLUMNS)

# COMMAND ----------

# Complete schema - ALL 21 columns from working scripts
person_schema = StructType([
    StructField("person_id", IntegerType(), True),
    StructField("gender_concept_id", IntegerType(), True),
    StructField("year_of_birth", IntegerType(), True),
    StructField("month_of_birth", IntegerType(), True),
    StructField("day_of_birth", IntegerType(), True),
    StructField("birth_datetime", DateType(), True),
    StructField("race_concept_id", IntegerType(), True),
    StructField("ethnicity_concept_id", IntegerType(), True),
    StructField("location_id", StringType(), True),
    StructField("provider_id", StringType(), True),
    StructField("care_site_id", StringType(), True),
    StructField("person_source_value", StringType(), True),
    StructField("gender_source_value", StringType(), True),
    StructField("gender_source_concept_id", IntegerType(), True),
    StructField("race_source_value", StringType(), True),
    StructField("race_source_concept_id", IntegerType(), True),
    StructField("ethnicity_source_value", StringType(), True),
    StructField("ethnicity_source_concept_id", IntegerType(), True),
    StructField("created_timestamp", TimestampType(), True),
    StructField("updated_timestamp", TimestampType(), True),
    StructField("is_deleted", BooleanType(), True)
])

print("✅ Schema loaded - 21 columns (matching INSERT_3_VALID_RECORDS)")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ⏰ TIMESTAMP (FUTURE - MATCHING WORKING SCRIPTS)

# COMMAND ----------

# EXACT match with INSERT_3_VALID_RECORDS - future timestamp
base_timestamp = datetime.now() + timedelta(minutes=Config.FUTURE_MINUTES)
print(f"📅 Using future timestamp (+{Config.FUTURE_MINUTES} min): {base_timestamp}")
print(f"   (This ensures incremental load picks up these records)")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 👤 NAME POOLS (FOR REALISTIC NAMES)

# COMMAND ----------

# Expanded name pools for variety
FIRST_NAMES_MALE = [
    "James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph",
    "Thomas", "Charles", "Christopher", "Daniel", "Matthew", "Anthony", "Donald",
    "Mark", "Paul", "Steven", "Andrew", "Kenneth", "Joshua", "Kevin", "Brian",
    "George", "Edward", "Ronald", "Timothy", "Jason", "Jeffrey", "Ryan"
]

FIRST_NAMES_FEMALE = [
    "Mary", "Patricia", "Jennifer", "Linda", "Barbara", "Susan", "Jessica",
    "Sarah", "Karen", "Nancy", "Lisa", "Betty", "Margaret", "Sandra",
    "Ashley", "Kimberly", "Emily", "Donna", "Michelle", "Carol", "Amanda",
    "Melissa", "Deborah", "Stephanie", "Dorothy", "Rebecca", "Sharon", "Laura"
]

LAST_NAMES = [
    "Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller",
    "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez",
    "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin",
    "Lee", "Perez", "Thompson", "White", "Harris", "Sanchez", "Clark"
]

print(f"✅ Name pools loaded:")
print(f"   Male names:   {len(FIRST_NAMES_MALE)}")
print(f"   Female names: {len(FIRST_NAMES_FEMALE)}")
print(f"   Last names:   {len(LAST_NAMES)}")

# Create broadcast DataFrames for fast lookup
male_names_df = spark.createDataFrame(
    [(i, name) for i, name in enumerate(FIRST_NAMES_MALE)],
    ["name_idx", "first_name"]
)

female_names_df = spark.createDataFrame(
    [(i, name) for i, name in enumerate(FIRST_NAMES_FEMALE)],
    ["name_idx", "first_name"]
)

last_names_df = spark.createDataFrame(
    [(i, name) for i, name in enumerate(LAST_NAMES)],
    ["last_idx", "last_name"]
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 📊 TABLE ANALYSIS

# COMMAND ----------

print("\n🔍 TABLE ANALYSIS:")
print("-" * 80)

# Check if table exists
table_exists = spark.catalog.tableExists(Config.TARGET_TABLE)

if table_exists:
    target_df = spark.table(Config.TARGET_TABLE)
    stats = target_df.select(
        F.count("*").alias("current_count"),
        F.max("person_id").alias("max_id")
    ).collect()[0]
    
    current_count = stats["current_count"]
    max_id = stats["max_id"]
    
    print(f"   Current records: {current_count:,}")
    print(f"   Current max ID:  {max_id or 0:,}")
else:
    current_count = 0
    max_id = None
    print(f"   Table doesn't exist - will create new")

print("-" * 80)

# COMMAND ----------

# MAGIC %md
# MAGIC ## ✅ GENERATE CLEAN RECORDS (ALL 21 COLUMNS)

# COMMAND ----------

print(f"\n✅ GENERATING {Config.CLEAN_COUNT:,} CLEAN RECORDS...")
print("-" * 80)
clean_start = time.time()

if Config.CLEAN_COUNT > 0:
    # Base DataFrame with IDs
    clean_df = spark.range(0, Config.CLEAN_COUNT) \
        .withColumn("person_id", (F.lit(Config.START_ID) + F.col("id")).cast(IntegerType()))
    
    current_year = datetime.now().year
    
    # ===== GENDER ASSIGNMENT =====
    gender_rand = F.rand()
    clean_df = clean_df.withColumn("gender_concept_id",
        F.when(gender_rand < Config.FEMALE_PERCENT/100, 8532)  # Female
         .when(gender_rand < (Config.FEMALE_PERCENT + Config.MALE_PERCENT)/100, 8507)  # Male
         .otherwise(8551)  # Unknown
         .cast(IntegerType()))
    
    # ===== AGE ASSIGNMENT =====
    age_rand = F.rand()
    clean_df = clean_df.withColumn("age_group",
        F.when(age_rand < Config.BABY_PERCENT/100, "baby")
         .when(age_rand < (Config.BABY_PERCENT + Config.CHILD_PERCENT)/100, "child")
         .when(age_rand < (Config.BABY_PERCENT + Config.CHILD_PERCENT + Config.TEEN_PERCENT)/100, "teen")
         .when(age_rand < (Config.BABY_PERCENT + Config.CHILD_PERCENT + Config.TEEN_PERCENT + Config.YOUNG_ADULT_PERCENT)/100, "young_adult")
         .when(age_rand < (Config.BABY_PERCENT + Config.CHILD_PERCENT + Config.TEEN_PERCENT + Config.YOUNG_ADULT_PERCENT + Config.ADULT_PERCENT)/100, "adult")
         .otherwise("senior"))
    
    # Assign actual ages
    clean_df = clean_df.withColumn("age_years",
        F.when(F.col("age_group") == "baby", (F.rand() * 3).cast(IntegerType()))
         .when(F.col("age_group") == "child", 3 + (F.rand() * 10).cast(IntegerType()))
         .when(F.col("age_group") == "teen", 13 + (F.rand() * 7).cast(IntegerType()))
         .when(F.col("age_group") == "young_adult", 20 + (F.rand() * 16).cast(IntegerType()))
         .when(F.col("age_group") == "adult", 36 + (F.rand() * 29).cast(IntegerType()))
         .otherwise(65 + (F.rand() * 25).cast(IntegerType())))
    
    # Calculate birth year
    clean_df = clean_df.withColumn("year_of_birth", 
        F.lit(current_year) - F.col("age_years"))
    
    # Month and day
    clean_df = clean_df.withColumn("month_of_birth", 
        (F.rand() * 11 + 1).cast(IntegerType())) \
        .withColumn("day_of_birth",
        F.when(F.col("month_of_birth").isin(4, 6, 9, 11), (F.rand() * 29 + 1).cast(IntegerType()))
         .when(F.col("month_of_birth") == 2, (F.rand() * 27 + 1).cast(IntegerType()))
         .otherwise((F.rand() * 30 + 1).cast(IntegerType())))
    
    # Birth datetime
    clean_df = clean_df.withColumn("birth_datetime",
        F.to_date(F.concat_ws("-", F.col("year_of_birth"), F.col("month_of_birth"), F.col("day_of_birth"))))
    
    # ===== RACE AND ETHNICITY =====
    race_rand = F.rand()
    clean_df = clean_df.withColumn("race_concept_id",
        F.when(race_rand < 0.60, 8527)   # White
         .when(race_rand < 0.75, 8516)   # Black
         .when(race_rand < 0.87, 8515)   # Asian
         .otherwise(0)                    # Other/Unknown
         .cast(IntegerType()))
    
    ethnic_rand = F.rand()
    clean_df = clean_df.withColumn("ethnicity_concept_id",
        F.when(ethnic_rand < 0.82, 38003564)  # Not Hispanic
         .otherwise(38003563)                  # Hispanic
         .cast(IntegerType()))
    
    # ===== LOCATION/PROVIDER/CARE SITE (matching INSERT_3_VALID_RECORDS) =====
    clean_df = clean_df.withColumn("location_id",
        F.concat(F.lit("LOC_VALID_"), F.lpad(F.col("id").cast("string"), 3, "0"))) \
        .withColumn("provider_id",
        F.concat(F.lit("PROV_VALID_"), F.lpad(F.col("id").cast("string"), 3, "0"))) \
        .withColumn("care_site_id",
        F.concat(F.lit("SITE_VALID_"), F.lpad(F.col("id").cast("string"), 3, "0")))
    
    # ===== REALISTIC NAMES =====
    # Generate random indices
    clean_df = clean_df.withColumn("first_idx", 
        F.when(F.col("gender_concept_id") == 8507,
               (F.rand() * len(FIRST_NAMES_MALE)).cast(IntegerType()))
         .when(F.col("gender_concept_id") == 8532,
               (F.rand() * len(FIRST_NAMES_FEMALE)).cast(IntegerType()))
         .otherwise((F.rand() * len(FIRST_NAMES_MALE)).cast(IntegerType()))) \
        .withColumn("last_idx", (F.rand() * len(LAST_NAMES)).cast(IntegerType()))
    
    # Join for male names
    male_df = clean_df.filter(F.col("gender_concept_id") == 8507) \
        .join(F.broadcast(male_names_df), clean_df["first_idx"] == male_names_df["name_idx"], "left") \
        .drop("name_idx")
    
    # Join for female names
    female_df = clean_df.filter(F.col("gender_concept_id") == 8532) \
        .join(F.broadcast(female_names_df), clean_df["first_idx"] == female_names_df["name_idx"], "left") \
        .drop("name_idx")
    
    # Join for unknown gender
    unknown_df = clean_df.filter(~F.col("gender_concept_id").isin(8507, 8532)) \
        .join(F.broadcast(male_names_df), clean_df["first_idx"] == male_names_df["name_idx"], "left") \
        .drop("name_idx")
    
    # Union and join last names
    clean_df = male_df.unionByName(female_df, allowMissingColumns=True) \
                     .unionByName(unknown_df, allowMissingColumns=True) \
                     .join(F.broadcast(last_names_df), clean_df["last_idx"] == last_names_df["last_idx"], "left") \
                     .drop("last_idx")
    
    # Create person_source_value (full name)
    clean_df = clean_df.withColumn("person_source_value",
        F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
    
    # ===== GENDER SOURCE VALUES =====
    clean_df = clean_df.withColumn("gender_source_value",
        F.when(F.col("gender_concept_id") == 8507, F.lit("M"))
         .when(F.col("gender_concept_id") == 8532, F.lit("F"))
         .otherwise(F.lit("U")))
    
    clean_df = clean_df.withColumn("gender_source_concept_id", F.col("gender_concept_id"))
    
    # ===== RACE/ETHNICITY SOURCE VALUES =====
    clean_df = clean_df.withColumn("race_source_value",
        F.when(F.col("race_concept_id") == 8527, F.lit("White"))
         .when(F.col("race_concept_id") == 8516, F.lit("Black"))
         .when(F.col("race_concept_id") == 8515, F.lit("Asian"))
         .otherwise(F.lit("Other")))
    
    clean_df = clean_df.withColumn("race_source_concept_id", F.col("race_concept_id"))
    
    clean_df = clean_df.withColumn("ethnicity_source_value",
        F.when(F.col("ethnicity_concept_id") == 38003564, F.lit("Not Hispanic"))
         .otherwise(F.lit("Hispanic")))
    
    clean_df = clean_df.withColumn("ethnicity_source_concept_id", F.col("ethnicity_concept_id"))
    
    # ===== TIMESTAMPS (future, matching INSERT_3_VALID_RECORDS) =====
    clean_df = clean_df.withColumn("created_timestamp", F.lit(base_timestamp)) \
        .withColumn("updated_timestamp", F.lit(base_timestamp)) \
        .withColumn("is_deleted", F.lit(False))
    
    # Drop temporary columns
    clean_df = clean_df.drop("id", "age_group", "age_years", "first_name", "last_name", "first_idx", "last_idx")
    
    clean_time = time.time() - clean_start
    print(f"   ✅ Generated {Config.CLEAN_COUNT:,} clean records in {clean_time:.2f}s")
    print(f"   ✅ ALL 21 columns populated")
else:
    clean_df = None
    print(f"   ⏭️ Skipping clean records (CLEAN_COUNT = 0)")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🚫 GENERATE RAW RECORDS (ALL 21 COLUMNS)

# COMMAND ----------

print(f"\n🚫 GENERATING {Config.RAW_COUNT:,} RAW RECORDS...")
print("-" * 80)
raw_start = time.time()

if Config.RAW_COUNT > 0:
    # Calculate counts for each raw type
    null_id_count = int(Config.RAW_COUNT * Config.NULL_ID_PERCENT / 100)
    bad_gender_count = int(Config.RAW_COUNT * Config.BAD_GENDER_PERCENT / 100)
    future_birth_count = int(Config.RAW_COUNT * Config.FUTURE_BIRTH_PERCENT / 100)
    old_year_count = int(Config.RAW_COUNT * Config.OLD_YEAR_PERCENT / 100)
    bad_month_count = int(Config.RAW_COUNT * Config.BAD_MONTH_PERCENT / 100)
    bad_day_count = int(Config.RAW_COUNT * Config.BAD_DAY_PERCENT / 100)
    
    # Adjust for rounding
    total_raw = (null_id_count + bad_gender_count + future_birth_count + 
                 old_year_count + bad_month_count + bad_day_count)
    if total_raw < Config.RAW_COUNT:
        remainder = Config.RAW_COUNT - total_raw
        null_id_count += remainder
        print(f"   ⚠️ Adjusted NULL_ID count by +{remainder} for rounding")
    
    print(f"   Breakdown by raw type:")
    print(f"     NULL ID:         {null_id_count:,}")
    print(f"     Bad Gender:      {bad_gender_count:,}")
    print(f"     Future Birth:    {future_birth_count:,}")
    print(f"     Old Year (<1900): {old_year_count:,}")
    print(f"     Bad Month (13-15): {bad_month_count:,}")
    print(f"     Bad Day (32-35):  {bad_day_count:,}")
    
    current_year = datetime.now().year
    raw_dfs = []
    current_offset = 0
    
    # ===== NULL ID RECORDS =====
    if null_id_count > 0:
        null_df = spark.range(0, null_id_count)
        
        null_df = null_df.withColumn("person_id", F.lit(None).cast(IntegerType())) \
            .withColumn("gender_concept_id", F.when(F.rand() < 0.5, 8507).otherwise(8532).cast(IntegerType())) \
            .withColumn("year_of_birth", F.lit(current_year - 30).cast(IntegerType())) \
            .withColumn("month_of_birth", (F.rand() * 11 + 1).cast(IntegerType())) \
            .withColumn("day_of_birth", (F.rand() * 27 + 1).cast(IntegerType())) \
            .withColumn("birth_datetime", 
                F.to_date(F.concat_ws("-", 
                    F.lit(current_year - 30), 
                    (F.rand() * 11 + 1).cast(IntegerType()), 
                    (F.rand() * 27 + 1).cast(IntegerType())))) \
            .withColumn("race_concept_id", 
                F.when(F.rand() < 0.60, 8527)
                 .when(F.rand() < 0.75, 8516)
                 .when(F.rand() < 0.87, 8515)
                 .otherwise(0).cast(IntegerType())) \
            .withColumn("ethnicity_concept_id", 
                F.when(F.rand() < 0.82, 38003564).otherwise(38003563).cast(IntegerType())) \
            .withColumn("location_id", F.concat(F.lit("LOC_RAW_"), F.lpad(F.col("id").cast("string"), 3, "0"))) \
            .withColumn("provider_id", F.concat(F.lit("PROV_RAW_"), F.lpad(F.col("id").cast("string"), 3, "0"))) \
            .withColumn("care_site_id", F.concat(F.lit("SITE_RAW_"), F.lpad(F.col("id").cast("string"), 3, "0"))) \
            .withColumn("person_source_value", F.concat(F.lit("RAW_NULL_ID_"), F.lpad(F.col("id").cast("string"), 3, "0"))) \
            .withColumn("gender_source_value", F.lit("U")) \
            .withColumn("gender_source_concept_id", F.col("gender_concept_id")) \
            .withColumn("race_source_value",
                F.when(F.col("race_concept_id") == 8527, F.lit("White"))
                 .when(F.col("race_concept_id") == 8516, F.lit("Black"))
                 .when(F.col("race_concept_id") == 8515, F.lit("Asian"))
                 .otherwise(F.lit("Other"))) \
            .withColumn("race_source_concept_id", F.col("race_concept_id")) \
            .withColumn("ethnicity_source_value",
                F.when(F.col("ethnicity_concept_id") == 38003564, F.lit("Not Hispanic"))
                 .otherwise(F.lit("Hispanic"))) \
            .withColumn("ethnicity_source_concept_id", F.col("ethnicity_concept_id")) \
            .withColumn("created_timestamp", F.lit(base_timestamp)) \
            .withColumn("updated_timestamp", F.lit(base_timestamp)) \
            .withColumn("is_deleted", F.lit(False))
        
        raw_dfs.append(null_df)
        current_offset += null_id_count
    
    # ===== BAD GENDER RECORDS =====
    if bad_gender_count > 0:
        bad_gender_df = spark.range(0, bad_gender_count) \
            .withColumn("person_id", (F.lit(Config.START_ID + Config.CLEAN_COUNT + current_offset) + F.col("id")).cast(IntegerType()))
        
        bad_gender_df = bad_gender_df \
            .withColumn("gender_concept_id", 
                F.when(F.rand() < 0.33, 7777)
                 .when(F.rand() < 0.66, 8888)
                 .otherwise(9999).cast(IntegerType())) \
            .withColumn("year_of_birth", F.lit(current_year - 30).cast(IntegerType())) \
            .withColumn("month_of_birth", (F.rand() * 11 + 1).cast(IntegerType())) \
            .withColumn("day_of_birth", (F.rand() * 27 + 1).cast(IntegerType())) \
            .withColumn("birth_datetime", 
                F.to_date(F.concat_ws("-", 
                    F.lit(current_year - 30), 
                    (F.rand() * 11 + 1).cast(IntegerType()), 
                    (F.rand() * 27 + 1).cast(IntegerType())))) \
            .withColumn("race_concept_id", 
                F.when(F.rand() < 0.60, 8527)
                 .when(F.rand() < 0.75, 8516)
                 .when(F.rand() < 0.87, 8515)
                 .otherwise(0).cast(IntegerType())) \
            .withColumn("ethnicity_concept_id", 
                F.when(F.rand() < 0.82, 38003564).otherwise(38003563).cast(IntegerType())) \
            .withColumn("location_id", F.concat(F.lit("LOC_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("provider_id", F.concat(F.lit("PROV_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("care_site_id", F.concat(F.lit("SITE_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("person_source_value", F.concat(F.lit("RAW_BAD_GENDER_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("gender_source_value", F.lit("X")) \
            .withColumn("gender_source_concept_id", F.col("gender_concept_id")) \
            .withColumn("race_source_value",
                F.when(F.col("race_concept_id") == 8527, F.lit("White"))
                 .when(F.col("race_concept_id") == 8516, F.lit("Black"))
                 .when(F.col("race_concept_id") == 8515, F.lit("Asian"))
                 .otherwise(F.lit("Other"))) \
            .withColumn("race_source_concept_id", F.col("race_concept_id")) \
            .withColumn("ethnicity_source_value",
                F.when(F.col("ethnicity_concept_id") == 38003564, F.lit("Not Hispanic"))
                 .otherwise(F.lit("Hispanic"))) \
            .withColumn("ethnicity_source_concept_id", F.col("ethnicity_concept_id")) \
            .withColumn("created_timestamp", F.lit(base_timestamp)) \
            .withColumn("updated_timestamp", F.lit(base_timestamp)) \
            .withColumn("is_deleted", F.lit(False))
        
        raw_dfs.append(bad_gender_df)
        current_offset += bad_gender_count
    
    # ===== FUTURE BIRTH RECORDS =====
    if future_birth_count > 0:
        future_birth_df = spark.range(0, future_birth_count) \
            .withColumn("person_id", (F.lit(Config.START_ID + Config.CLEAN_COUNT + current_offset) + F.col("id")).cast(IntegerType()))
        
        future_year = current_year + 5
        future_birth_df = future_birth_df \
            .withColumn("gender_concept_id", F.when(F.rand() < 0.5, 8507).otherwise(8532).cast(IntegerType())) \
            .withColumn("year_of_birth", F.lit(future_year + (F.rand() * 10).cast(IntegerType()))) \
            .withColumn("month_of_birth", (F.rand() * 11 + 1).cast(IntegerType())) \
            .withColumn("day_of_birth", (F.rand() * 27 + 1).cast(IntegerType())) \
            .withColumn("birth_datetime", 
                F.to_date(F.concat_ws("-", 
                    F.col("year_of_birth"), 
                    F.col("month_of_birth"), 
                    F.col("day_of_birth")))) \
            .withColumn("race_concept_id", 
                F.when(F.rand() < 0.60, 8527)
                 .when(F.rand() < 0.75, 8516)
                 .when(F.rand() < 0.87, 8515)
                 .otherwise(0).cast(IntegerType())) \
            .withColumn("ethnicity_concept_id", 
                F.when(F.rand() < 0.82, 38003564).otherwise(38003563).cast(IntegerType())) \
            .withColumn("location_id", F.concat(F.lit("LOC_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("provider_id", F.concat(F.lit("PROV_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("care_site_id", F.concat(F.lit("SITE_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("person_source_value", F.concat(F.lit("RAW_FUTURE_BIRTH_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("gender_source_value",
                F.when(F.col("gender_concept_id") == 8507, F.lit("M"))
                 .when(F.col("gender_concept_id") == 8532, F.lit("F"))
                 .otherwise(F.lit("U"))) \
            .withColumn("gender_source_concept_id", F.col("gender_concept_id")) \
            .withColumn("race_source_value",
                F.when(F.col("race_concept_id") == 8527, F.lit("White"))
                 .when(F.col("race_concept_id") == 8516, F.lit("Black"))
                 .when(F.col("race_concept_id") == 8515, F.lit("Asian"))
                 .otherwise(F.lit("Other"))) \
            .withColumn("race_source_concept_id", F.col("race_concept_id")) \
            .withColumn("ethnicity_source_value",
                F.when(F.col("ethnicity_concept_id") == 38003564, F.lit("Not Hispanic"))
                 .otherwise(F.lit("Hispanic"))) \
            .withColumn("ethnicity_source_concept_id", F.col("ethnicity_concept_id")) \
            .withColumn("created_timestamp", F.lit(base_timestamp)) \
            .withColumn("updated_timestamp", F.lit(base_timestamp)) \
            .withColumn("is_deleted", F.lit(False))
        
        raw_dfs.append(future_birth_df)
        current_offset += future_birth_count
    
    # ===== OLD YEAR (<1900) RECORDS =====
    if old_year_count > 0:
        old_year_df = spark.range(0, old_year_count) \
            .withColumn("person_id", (F.lit(Config.START_ID + Config.CLEAN_COUNT + current_offset) + F.col("id")).cast(IntegerType()))
        
        old_year_df = old_year_df \
            .withColumn("gender_concept_id", F.when(F.rand() < 0.5, 8507).otherwise(8532).cast(IntegerType())) \
            .withColumn("year_of_birth", F.lit(1700 + (F.rand() * 199).cast(IntegerType()))) \
            .withColumn("month_of_birth", (F.rand() * 11 + 1).cast(IntegerType())) \
            .withColumn("day_of_birth", (F.rand() * 27 + 1).cast(IntegerType())) \
            .withColumn("birth_datetime", 
                F.to_date(F.concat_ws("-", 
                    F.col("year_of_birth"), 
                    F.col("month_of_birth"), 
                    F.col("day_of_birth")))) \
            .withColumn("race_concept_id", 
                F.when(F.rand() < 0.60, 8527)
                 .when(F.rand() < 0.75, 8516)
                 .when(F.rand() < 0.87, 8515)
                 .otherwise(0).cast(IntegerType())) \
            .withColumn("ethnicity_concept_id", 
                F.when(F.rand() < 0.82, 38003564).otherwise(38003563).cast(IntegerType())) \
            .withColumn("location_id", F.concat(F.lit("LOC_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("provider_id", F.concat(F.lit("PROV_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("care_site_id", F.concat(F.lit("SITE_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("person_source_value", F.concat(F.lit("RAW_OLD_YEAR_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("gender_source_value",
                F.when(F.col("gender_concept_id") == 8507, F.lit("M"))
                 .when(F.col("gender_concept_id") == 8532, F.lit("F"))
                 .otherwise(F.lit("U"))) \
            .withColumn("gender_source_concept_id", F.col("gender_concept_id")) \
            .withColumn("race_source_value",
                F.when(F.col("race_concept_id") == 8527, F.lit("White"))
                 .when(F.col("race_concept_id") == 8516, F.lit("Black"))
                 .when(F.col("race_concept_id") == 8515, F.lit("Asian"))
                 .otherwise(F.lit("Other"))) \
            .withColumn("race_source_concept_id", F.col("race_concept_id")) \
            .withColumn("ethnicity_source_value",
                F.when(F.col("ethnicity_concept_id") == 38003564, F.lit("Not Hispanic"))
                 .otherwise(F.lit("Hispanic"))) \
            .withColumn("ethnicity_source_concept_id", F.col("ethnicity_concept_id")) \
            .withColumn("created_timestamp", F.lit(base_timestamp)) \
            .withColumn("updated_timestamp", F.lit(base_timestamp)) \
            .withColumn("is_deleted", F.lit(False))
        
        raw_dfs.append(old_year_df)
        current_offset += old_year_count
    
    # ===== BAD MONTH RECORDS =====
    if bad_month_count > 0:
        bad_month_df = spark.range(0, bad_month_count) \
            .withColumn("person_id", (F.lit(Config.START_ID + Config.CLEAN_COUNT + current_offset) + F.col("id")).cast(IntegerType()))
        
        bad_month_df = bad_month_df \
            .withColumn("gender_concept_id", F.when(F.rand() < 0.5, 8507).otherwise(8532).cast(IntegerType())) \
            .withColumn("year_of_birth", F.lit(current_year - 30).cast(IntegerType())) \
            .withColumn("month_of_birth", F.lit(13 + (F.rand() * 2).cast(IntegerType()))) \
            .withColumn("day_of_birth", (F.rand() * 27 + 1).cast(IntegerType())) \
            .withColumn("birth_datetime", 
                F.to_date(F.concat_ws("-", 
                    F.lit(current_year - 30), 
                    F.col("month_of_birth"), 
                    F.col("day_of_birth")))) \
            .withColumn("race_concept_id", 
                F.when(F.rand() < 0.60, 8527)
                 .when(F.rand() < 0.75, 8516)
                 .when(F.rand() < 0.87, 8515)
                 .otherwise(0).cast(IntegerType())) \
            .withColumn("ethnicity_concept_id", 
                F.when(F.rand() < 0.82, 38003564).otherwise(38003563).cast(IntegerType())) \
            .withColumn("location_id", F.concat(F.lit("LOC_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("provider_id", F.concat(F.lit("PROV_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("care_site_id", F.concat(F.lit("SITE_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("person_source_value", F.concat(F.lit("RAW_BAD_MONTH_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("gender_source_value",
                F.when(F.col("gender_concept_id") == 8507, F.lit("M"))
                 .when(F.col("gender_concept_id") == 8532, F.lit("F"))
                 .otherwise(F.lit("U"))) \
            .withColumn("gender_source_concept_id", F.col("gender_concept_id")) \
            .withColumn("race_source_value",
                F.when(F.col("race_concept_id") == 8527, F.lit("White"))
                 .when(F.col("race_concept_id") == 8516, F.lit("Black"))
                 .when(F.col("race_concept_id") == 8515, F.lit("Asian"))
                 .otherwise(F.lit("Other"))) \
            .withColumn("race_source_concept_id", F.col("race_concept_id")) \
            .withColumn("ethnicity_source_value",
                F.when(F.col("ethnicity_concept_id") == 38003564, F.lit("Not Hispanic"))
                 .otherwise(F.lit("Hispanic"))) \
            .withColumn("ethnicity_source_concept_id", F.col("ethnicity_concept_id")) \
            .withColumn("created_timestamp", F.lit(base_timestamp)) \
            .withColumn("updated_timestamp", F.lit(base_timestamp)) \
            .withColumn("is_deleted", F.lit(False))
        
        raw_dfs.append(bad_month_df)
        current_offset += bad_month_count
    
    # ===== BAD DAY RECORDS =====
    if bad_day_count > 0:
        bad_day_df = spark.range(0, bad_day_count) \
            .withColumn("person_id", (F.lit(Config.START_ID + Config.CLEAN_COUNT + current_offset) + F.col("id")).cast(IntegerType()))
        
        bad_day_df = bad_day_df \
            .withColumn("gender_concept_id", F.when(F.rand() < 0.5, 8507).otherwise(8532).cast(IntegerType())) \
            .withColumn("year_of_birth", F.lit(current_year - 30).cast(IntegerType())) \
            .withColumn("month_of_birth", (F.rand() * 11 + 1).cast(IntegerType())) \
            .withColumn("day_of_birth", F.lit(32 + (F.rand() * 3).cast(IntegerType()))) \
            .withColumn("birth_datetime", 
                F.to_date(F.concat_ws("-", 
                    F.lit(current_year - 30), 
                    F.col("month_of_birth"), 
                    F.col("day_of_birth")))) \
            .withColumn("race_concept_id", 
                F.when(F.rand() < 0.60, 8527)
                 .when(F.rand() < 0.75, 8516)
                 .when(F.rand() < 0.87, 8515)
                 .otherwise(0).cast(IntegerType())) \
            .withColumn("ethnicity_concept_id", 
                F.when(F.rand() < 0.82, 38003564).otherwise(38003563).cast(IntegerType())) \
            .withColumn("location_id", F.concat(F.lit("LOC_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("provider_id", F.concat(F.lit("PROV_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("care_site_id", F.concat(F.lit("SITE_RAW_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("person_source_value", F.concat(F.lit("RAW_BAD_DAY_"), F.lpad((F.lit(current_offset) + F.col("id")).cast("string"), 3, "0"))) \
            .withColumn("gender_source_value",
                F.when(F.col("gender_concept_id") == 8507, F.lit("M"))
                 .when(F.col("gender_concept_id") == 8532, F.lit("F"))
                 .otherwise(F.lit("U"))) \
            .withColumn("gender_source_concept_id", F.col("gender_concept_id")) \
            .withColumn("race_source_value",
                F.when(F.col("race_concept_id") == 8527, F.lit("White"))
                 .when(F.col("race_concept_id") == 8516, F.lit("Black"))
                 .when(F.col("race_concept_id") == 8515, F.lit("Asian"))
                 .otherwise(F.lit("Other"))) \
            .withColumn("race_source_concept_id", F.col("race_concept_id")) \
            .withColumn("ethnicity_source_value",
                F.when(F.col("ethnicity_concept_id") == 38003564, F.lit("Not Hispanic"))
                 .otherwise(F.lit("Hispanic"))) \
            .withColumn("ethnicity_source_concept_id", F.col("ethnicity_concept_id")) \
            .withColumn("created_timestamp", F.lit(base_timestamp)) \
            .withColumn("updated_timestamp", F.lit(base_timestamp)) \
            .withColumn("is_deleted", F.lit(False))
        
        raw_dfs.append(bad_day_df)
        current_offset += bad_day_count
    
    # Union all raw DataFrames
    raw_df = raw_dfs[0]
    for df in raw_dfs[1:]:
        raw_df = raw_df.unionByName(df, allowMissingColumns=True)
    
    raw_time = time.time() - raw_start
    print(f"   ✅ Generated {Config.RAW_COUNT:,} raw records in {raw_time:.2f}s")
    print(f"   ✅ ALL 21 columns populated")
else:
    raw_df = None
    print(f"   ⏭️ Skipping raw records (RAW_COUNT = 0)")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🔗 COMBINE AND ALIGN SCHEMA

# COMMAND ----------

print(f"\n🔧 COMBINING AND ALIGNING SCHEMA...")
combine_start = time.time()

# Combine clean and raw DataFrames
dfs_to_union = []
if clean_df is not None:
    dfs_to_union.append(clean_df)
if raw_df is not None:
    dfs_to_union.append(raw_df)

if dfs_to_union:
    final_df = dfs_to_union[0]
    for df in dfs_to_union[1:]:
        final_df = final_df.unionByName(df, allowMissingColumns=True)
    
    # Ensure all schema fields exist in correct order
    aligned_columns = []
    for field in person_schema.fields:
        if field.name in final_df.columns:
            aligned_columns.append(F.col(field.name).cast(field.dataType))
        else:
            aligned_columns.append(F.lit(None).cast(field.dataType).alias(field.name))
    
    final_df = final_df.select(aligned_columns)
    
    combine_time = time.time() - combine_start
    print(f"   ✅ Combined and aligned {total_records:,} records in {combine_time:.2f}s")
    print(f"   ✅ Schema matches INSERT_3_VALID_RECORDS exactly")
else:
    print(f"   ⚠️ No records to generate!")
    final_df = None

# COMMAND ----------

# MAGIC %md
# MAGIC ## 💾 WRITE TO TABLE

# COMMAND ----------

if final_df is not None:
    print(f"\n💾 WRITING TO {Config.TARGET_TABLE}...")
    print("-" * 80)
    write_start = time.time()
    
    try:
        final_df.write.format("delta").mode("append").saveAsTable(Config.TARGET_TABLE)
        write_time = time.time() - write_start
        print(f"   ✅ Write completed: {write_time:.2f}s")
    except Exception as e:
        print(f"   ❌ ERROR: {str(e)}")
        raise
    
    print("-" * 80)

# COMMAND ----------

# MAGIC %md
# MAGIC ## ✅ VERIFICATION

# COMMAND ----------

if final_df is not None:
    print(f"\n✅ VERIFICATION:")
    print("-" * 80)
    
    # Get final count
    final_count = spark.table(Config.TARGET_TABLE).count()
    expected = current_count + total_records
    
    print(f"   Initial count:  {current_count:,}")
    print(f"   Records added:  {total_records:,}")
    print(f"   Expected total: {expected:,}")
    print(f"   Actual total:   {final_count:,}")
    print(f"   Status:         {'✅ PASS' if final_count == expected else '❌ FAIL'}")
    
    # Show sample of clean records
    if clean_df is not None:
        print(f"\n   📋 Sample of CLEAN records:")
        spark.table(Config.TARGET_TABLE) \
            .filter(F.col("person_source_value").rlike("^[A-Z][a-z]+ [A-Z][a-z]+$")) \
            .select("person_id", "person_source_value", "gender_concept_id", 
                    "year_of_birth", "location_id", "provider_id", "care_site_id") \
            .limit(3) \
            .show(truncate=False)
    
    # Show sample of raw records
    if raw_df is not None:
        print(f"\n   📋 Sample of RAW records:")
        spark.table(Config.TARGET_TABLE) \
            .filter(F.col("person_source_value").contains("RAW_")) \
            .select("person_id", "person_source_value", "gender_concept_id", 
                    "year_of_birth", "month_of_birth", "day_of_birth") \
            .limit(5) \
            .show(truncate=False)
    
    print("-" * 80)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 📊 SUMMARY

# COMMAND ----------

if final_df is not None:
    total_time = time.time() - clean_start if Config.CLEAN_COUNT > 0 else time.time() - raw_start
    
    print("\n" + "=" * 80)
    print("📊 GENERATION SUMMARY")
    print("=" * 80)
    print(f"Generator:      v5.1 CLEAN+RAW")
    print(f"Date:           {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Target Table:   {Config.TARGET_TABLE}")
    print(f"\n📊 RECORDS GENERATED:")
    print(f"   Clean:       {Config.CLEAN_COUNT:,}")
    print(f"   Raw:         {Config.RAW_COUNT:,}")
    print(f"   TOTAL:       {total_records:,}")
    print(f"\n⚡ PERFORMANCE:")
    print(f"   Generation:  {total_time:.2f}s")
    print(f"   Write:       {write_time:.2f}s")
    print(f"   Total:       {total_time + write_time:.2f}s")
    print(f"   Throughput:  {total_records / (total_time + write_time):,.0f} rows/sec")
    print(f"\n✅ SCHEMA VALIDATION:")
    print(f"   All 21 columns:    ✅")
    print(f"   Matches INSERT_3_VALID_RECORDS: ✅")
    print("=" * 80)
    
    # Expected quarantine results
    if Config.RAW_COUNT > 0:
        raw_quarantine = (null_id_count + bad_gender_count + future_birth_count + 
                         old_year_count + bad_month_count + bad_day_count)
        print(f"\n🔍 EXPECTED QUARANTINE RESULTS:")
        print(f"   Total records to quarantine: {raw_quarantine:,}")
        print(f"   - NULL ID:         {null_id_count:,} (NOT_NULL violation)")
        print(f"   - Bad Gender:      {bad_gender_count:,} (VALID_GENDER violation)")
        print(f"   - Future Birth:    {future_birth_count:,} (YEAR range violation)")
        print(f"   - Old Year (<1900): {old_year_count:,} (YEAR range violation)")
        print(f"   - Bad Month:       {bad_month_count:,} (MONTH range violation)")
        print(f"   - Bad Day:         {bad_day_count:,} (DAY range violation)")
        print("=" * 80)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🔧 USAGE INSTRUCTIONS
# MAGIC 
# MAGIC ### **How to use this generator:**
# MAGIC 
# MAGIC 1. **Change the numbers in CONFIGURATION:**
# MAGIC    ```python
# MAGIC    CLEAN_COUNT = 100     # Number of good records
# MAGIC    RAW_COUNT = 30        # Number of bad records
# MAGIC    ```
# MAGIC 
# MAGIC 2. **Adjust percentages as needed:**
# MAGIC    - Gender distribution (female/male/unknown)
# MAGIC    - Age distribution (babies to seniors)
# MAGIC    - Raw record types (what kinds of errors)
# MAGIC 
# MAGIC 3. **Run the notebook**
# MAGIC 
# MAGIC 4. **Run your ETL pipeline** to see quarantine in action
# MAGIC 
# MAGIC ### **Example Configurations:**
# MAGIC 
# MAGIC | Test Scenario | CLEAN_COUNT | RAW_COUNT | Notes |
# MAGIC |--------------|-------------|-----------|-------|
# MAGIC | Quick test | 10 | 3 | Small dataset |
# MAGIC | Balanced test | 100 | 30 | 77% clean, 23% raw |
# MAGIC | Quarantine test | 0 | 50 | Only raw records |
# MAGIC | Load test | 10,000 | 3,000 | Large dataset |
# MAGIC | Gender test | 100 | 0 | Only clean, check distribution |

StatementMeta(, 53ed2ae2-c8f3-4ec5-acfc-bf07e16ac3d9, 3, Finished, Available, Finished, False)

🚀 SYNTHETIC DATA GENERATOR v5.1 - CLEAN + RAW
Target: EXACT match with INSERT_3_VALID_RECORDS.py

📋 CONFIGURATION SUMMARY:
   Target Table:      dbo.person
   CLEAN Records:     500
   RAW Records:       30
   TOTAL Records:     530
   START_ID:          2,000,000
✅ Schema loaded - 21 columns (matching INSERT_3_VALID_RECORDS)
📅 Using future timestamp (+15 min): 2026-03-13 11:47:19.228719
   (This ensures incremental load picks up these records)
✅ Name pools loaded:
   Male names:   30
   Female names: 28
   Last names:   27

🔍 TABLE ANALYSIS:
--------------------------------------------------------------------------------
   Current records: 11,633,719
   Current max ID:  10,000,044
--------------------------------------------------------------------------------

✅ GENERATING 500 CLEAN RECORDS...
--------------------------------------------------------------------------------
   ✅ Generated 500 clean records in 1.23s
   ✅ ALL 21 columns populated

🚫 GENERATING 30 RAW RECORDS...
-------